<a href="https://colab.research.google.com/github/lvbchou/AIMS/blob/main/Intelligent_Memory_Management_System(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque, defaultdict
import random
import time
from datetime import datetime, timedelta
import threading
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

PHASE 1: CORE MEMORY MANAGEMENT STRUCTURES

In [ ]:
class PageTableEntry:
    """Implement Page Table Entry từ slide - Paging Strategy"""
    def __init__(self, frame_number=None):
        self.frame_number = frame_number  # Physical frame number
        self.present = frame_number is not None
        self.dirty = False               # Modified bit
        self.accessed = False           # Referenced bit
        self.protection = 'RW'          # Protection bits
        self.cache_disable = False      # Cache control
        self.last_access_time = 0      # For LRU algorithm

class PageTable:
    """Page Control Block (PCB) từ slide"""
    def __init__(self, num_pages, page_size=4096):
        self.num_pages = num_pages
        self.page_size = page_size
        self.entries = [PageTableEntry() for _ in range(num_pages)]
        self.tlb = {}  # Translation Lookaside Buffer
        self.tlb_hits = 0
        self.tlb_misses = 0

    def get_physical_address(self, virtual_address):
        """Convert virtual address to physical address"""
        page_number = virtual_address // self.page_size
        offset = virtual_address % self.page_size

        # Page validity check
        if page_number >= self.num_pages:
            raise Exception("Page fault - invalid page number")

        # Check TLB first
        if page_number in self.tlb:
            self.tlb_hits += 1
            frame_number = self.tlb[page_number]
        else:
            self.tlb_misses += 1
            entry = self.entries[page_number]

            if not entry.present:
                raise Exception("Page fault - page not in memory")

            frame_number = entry.frame_number
            # Update TLB
            if len(self.tlb) >= 64:  # TLB size limit
                # Remove oldest entry
                oldest_page = min(self.tlb.keys())
                del self.tlb[oldest_page]
            self.tlb[page_number] = frame_number

        # Update access information
        self.entries[page_number].accessed = True
        self.entries[page_number].last_access_time = time.time()

        return frame_number * self.page_size + offset

PHASE 2: PAGE REPLACEMENT ALGORITHMS

In [ ]:
class PageReplacementAlgorithm:
    """Base class for page replacement algorithms"""
    def __init__(self, num_frames):
        self.num_frames = num_frames
        self.frames = [-1] * num_frames  # -1 means empty frame
        self.page_faults = 0
        self.page_hits = 0

    def access_page(self, page_number):
        raise NotImplementedError

class FIFOPageReplacement(PageReplacementAlgorithm):
    """First In First Out - từ slide"""
    def __init__(self, num_frames):
        super().__init__(num_frames)
        self.queue = deque()

    def access_page(self, page_number):
        if page_number in self.frames:
            self.page_hits += 1
            return True, f"Page {page_number} hit"

        # Page fault occurred
        self.page_faults += 1

        if len(self.queue) < self.num_frames:
            # Frame available
            frame_index = len(self.queue)
            self.frames[frame_index] = page_number
            self.queue.append(page_number)
        else:
            # Replace oldest page
            oldest_page = self.queue.popleft()
            frame_index = self.frames.index(oldest_page)
            self.frames[frame_index] = page_number
            self.queue.append(page_number)

        return False, f"Page fault: loaded page {page_number}"

class LRUPageReplacement(PageReplacementAlgorithm):
    """Least Recently Used - từ slide"""
    def __init__(self, num_frames):
        super().__init__(num_frames)
        self.access_times = {}
        self.current_time = 0

    def access_page(self, page_number):
        self.current_time += 1

        if page_number in self.frames:
            self.page_hits += 1
            self.access_times[page_number] = self.current_time
            return True, f"Page {page_number} hit"

        # Page fault occurred
        self.page_faults += 1

        if -1 in self.frames:
            # Frame available
            frame_index = self.frames.index(-1)
            self.frames[frame_index] = page_number
            self.access_times[page_number] = self.current_time
        else:
            # Find LRU page
            lru_page = min(self.access_times.keys(),
                          key=lambda x: self.access_times[x] if x in self.frames else float('inf'))
            frame_index = self.frames.index(lru_page)

            # Remove old page and add new one
            del self.access_times[lru_page]
            self.frames[frame_index] = page_number
            self.access_times[page_number] = self.current_time

        return False, f"Page fault: loaded page {page_number}"

class OPTPageReplacement(PageReplacementAlgorithm):
    """Optimal Page Replacement - từ slide"""
    def __init__(self, num_frames, future_references):
        super().__init__(num_frames)
        self.future_references = future_references
        self.current_index = 0

    def access_page(self, page_number):
        if page_number in self.frames:
            self.page_hits += 1
            self.current_index += 1
            return True, f"Page {page_number} hit"

        # Page fault occurred
        self.page_faults += 1

        if -1 in self.frames:
            # Frame available
            frame_index = self.frames.index(-1)
            self.frames[frame_index] = page_number
        else:
            # Find optimal page to replace
            farthest_use = -1
            victim_frame = 0

            for i, frame_page in enumerate(self.frames):
                # Find next use of this page
                next_use = float('inf')
                for j in range(self.current_index + 1, len(self.future_references)):
                    if self.future_references[j] == frame_page:
                        next_use = j
                        break

                if next_use > farthest_use:
                    farthest_use = next_use
                    victim_frame = i

            self.frames[victim_frame] = page_number

        self.current_index += 1
        return False, f"Page fault: loaded page {page_number}"

PHASE 3: PROCESS SIMULATION WITH MEMORY PATTERNS

In [ ]:
class Process:
    """Enhanced Process class with realistic memory patterns"""
    def __init__(self, pid, memory_base=1024, cpu_intensity=0.5):
        self.pid = pid
        self.memory_base = memory_base  # KB
        self.cpu_intensity = cpu_intensity
        self.memory_current = memory_base
        self.lifecycle_stage = "startup"
        self.priority = random.randint(1, 5)
        #self.segment_table = SegmentTable()
        self.page_table = PageTable(num_pages=memory_base//4)  # 4KB pages

        # Memory pattern characteristics
        self.pattern_type = random.choice(['linear', 'periodic', 'burst', 'random_walk'])
        self.pattern_params = self._generate_pattern_params()

        # Usage history for ML features
        self.memory_history = deque(maxlen=100)
        self.cpu_history = deque(maxlen=100)
        self.created_time = datetime.now()

        # Create initial segments
        #self._create_initial_segments()

    def _generate_pattern_params(self):
        """Generate parameters for different memory patterns"""
        if self.pattern_type == 'linear':
            return {'growth_rate': random.uniform(0.01, 0.05)}
        elif self.pattern_type == 'periodic':
            return {
                'amplitude': random.uniform(0.2, 0.5),
                'period': random.randint(10, 50),
                'phase': random.uniform(0, 2*np.pi)
            }
        elif self.pattern_type == 'burst':
            return {
                'burst_probability': 0.05,
                'burst_magnitude': random.uniform(1.5, 3.0),
                'decay_rate': 0.95
            }
        else:  # random_walk
            return {
                'volatility': random.uniform(0.01, 0.1),
                'drift': random.uniform(-0.001, 0.001)
            }


    def simulate_memory_usage(self, time_step):
        """Simulate memory usage based on pattern and lifecycle"""
        base_memory = self.memory_base

        # Apply lifecycle stage effect
        lifecycle_multipliers = { #Nhân với memory_base để điều chỉnh bộ nhớ cơ sở
            'startup': 1.2,
            'active': 1.0,
            'idle': 0.7,
            'cleanup': 0.5
        }
        base_memory *= lifecycle_multipliers.get(self.lifecycle_stage, 1.0)

        # Apply pattern-specific changes
        if self.pattern_type == 'linear':
            growth = self.pattern_params['growth_rate'] * time_step
            self.memory_current = base_memory * (1 + growth)

        elif self.pattern_type == 'periodic':
            params = self.pattern_params
            periodic_factor = 1 + params['amplitude'] * np.sin(
                2 * np.pi * time_step / params['period'] + params['phase']
            )
            self.memory_current = base_memory * periodic_factor

        elif self.pattern_type == 'burst':
            if random.random() < self.pattern_params['burst_probability']:
                self.memory_current *= self.pattern_params['burst_magnitude']
            else:
                self.memory_current *= self.pattern_params['decay_rate']
            self.memory_current = max(self.memory_current, base_memory * 0.5)

        else:  # random_walk
            params = self.pattern_params
            change = np.random.normal(params['drift'], params['volatility'])
            self.memory_current *= (1 + change)
            self.memory_current = max(self.memory_current, base_memory * 0.3)

        # Update history
        self.memory_history.append(self.memory_current)
        self.cpu_history.append(self.cpu_intensity + random.uniform(-0.1, 0.1))

        # Update lifecycle stage
        self._update_lifecycle_stage(time_step)

        return self.memory_current

    def _update_lifecycle_stage(self, time_step):
        """Update process lifecycle stage"""
        age = time_step
        if age < 10:
            self.lifecycle_stage = "startup"
        elif age < 100:
            self.lifecycle_stage = "active"
        elif age < 200:
            self.lifecycle_stage = "idle"
        else:
            self.lifecycle_stage = "cleanup"

    def get_memory_features(self):
        """Extract features for ML prediction"""
        if len(self.memory_history) < 5:
            return None

        history = list(self.memory_history)
        features = {
            'memory_current': history[-1] if history else self.memory_base,
            'memory_avg_5': np.mean(history[-5:]) if len(history) >= 5 else self.memory_base,
            'memory_avg_10': np.mean(history[-10:]) if len(history) >= 10 else self.memory_base,
            'memory_trend': (history[-1] - history[-5]) / 5 if len(history) >= 5 else 0,
            'memory_volatility': np.std(history[-10:]) if len(history) >= 10 else 0,
            'cpu_avg': np.mean(list(self.cpu_history)[-10:]) if self.cpu_history else self.cpu_intensity,
            'lifecycle_startup': 1 if self.lifecycle_stage == 'startup' else 0,
            'lifecycle_active': 1 if self.lifecycle_stage == 'active' else 0,
            'lifecycle_idle': 1 if self.lifecycle_stage == 'idle' else 0,
            'lifecycle_cleanup': 1 if self.lifecycle_stage == 'cleanup' else 0,
            'priority': self.priority,
            'pattern_linear': 1 if self.pattern_type == 'linear' else 0,
            'pattern_periodic': 1 if self.pattern_type == 'periodic' else 0,
            'pattern_burst': 1 if self.pattern_type == 'burst' else 0,
            'pattern_random': 1 if self.pattern_type == 'random_walk' else 0
        }
        return features


PHASE 4: AI-POWERED MEMORY PREDICTOR

In [ ]:
class MemoryPredictor:
    """Machine Learning based memory usage predictor"""
    def __init__(self):
        self.model = LinearRegression()
        self.trained = False
        self.feature_columns = []
        self.training_data = []
        self.performance_metrics = {}

    def collect_training_data(self, processes, time_steps=200):
        """Collect training data from process simulations"""
        print("Collecting training data...")

        for step in range(time_steps):
            for process in processes:
                # Simulate one step
                actual_memory = process.simulate_memory_usage(step)

                # Get features
                features = process.get_memory_features()
                if features is not None:
                    # Add target (next step memory prediction)
                    next_memory = process.simulate_memory_usage(step + 1)
                    features['target'] = next_memory
                    self.training_data.append(features)

        print(f"Collected {len(self.training_data)} training samples")

    def train_models(self):
        """Train ML model on collected data"""
        if not self.training_data:
            raise ValueError("No training data available")

        print("Training Linear Regression model...")

        # Convert to DataFrame
        df = pd.DataFrame(self.training_data)

        # Prepare features and target
        target_col = 'target'
        self.feature_columns = [col for col in df.columns if col != target_col]

        X = df[self.feature_columns]
        y = df[target_col]

        # Split data
        split_idx = int(0.8 * len(X))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        # Train models
        self.model.fit(X_train, y_train)

        # Evaluate
        y_pred = self.model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        self.performance_metrics['linear'] = {
            'MAE': mae,
            'RMSE': rmse,
            'Accuracy_10pct': np.mean(np.abs(y_pred - y_test) / y_test < 0.1) * 100
        }

        print(f"Linear Regression: MAE={mae:.2f}, RMSE={rmse:.2f}, Accuracy±10%={self.performance_metrics['linear']['Accuracy_10pct']:.1f}%")

        self.trained = True
        print("Model training completed")

    def predict_memory_usage(self, process):
        """Predict future memory usage for a process"""
        if not self.trained:
            return process.memory_current  # Fallback to current usage

        features = process.get_memory_features()
        if features is None:
            return process.memory_current

        # Prepare features
        feature_values = [features.get(col, 0) for col in self.feature_columns]
        X = np.array(feature_values).reshape(1, -1)

        # Make prediction
        prediction = self.model.predict(X)[0]
        return max(prediction, process.memory_base * 0.1)  # Minimum threshold

PHASE 5: INTELLIGENT MEMORY MANAGER

In [ ]:
class IntelligentMemoryManager:
    """AI-powered Memory Manager with full OS theory implementation"""
    def __init__(self, total_memory=8192, page_size=4, num_frames=2048):  # 8GB, 4KB pages
        self.total_memory = total_memory  # KB
        self.page_size = page_size        # KB
        self.num_frames = num_frames

        # Physical memory management
        self.free_frames = set(range(num_frames))
        self.allocated_frames = {}  # pid -> set of frame numbers

        # Segmentation support
        self.global_segment_table = {}  # pid -> SegmentTable

        # Page replacement algorithms
        self.page_algorithms = {
            'FIFO': FIFOPageReplacement(num_frames // 4),
            'LRU': LRUPageReplacement(num_frames // 4),
            'OPT': None  # Will be initialized with reference string
        }

        # AI predictor
        self.predictor = MemoryPredictor()

        # Performance monitoring
        self.allocation_history = []
        self.performance_stats = {
            'total_allocations': 0,
            'successful_allocations': 0,
            'fragmentation_events': 0,
            'prediction_accuracy': []
        }

        # Process management
        self.processes = {}
        self.allocation_strategies = ['first_fit', 'best_fit', 'worst_fit', 'ai_optimized']
        self.current_strategy = 'ai_optimized'


    def register_process(self, process):
        """Register a new process with the memory manager"""
        self.processes[process.pid] = process
        #self.global_segment_table[process.pid] = process.segment_table
        self.allocated_frames[process.pid] = set()

        print(f"Registered process {process.pid} with pattern: {process.pattern_type}")

    def allocate_memory(self, pid, requested_memory, allocation_type='dynamic'):
        """Intelligent memory allocation"""
        self.performance_stats['total_allocations'] += 1

        if pid not in self.processes:
            return False, "Process not registered"

        process = self.processes[pid]

        # AI-based prediction for proactive allocation
        if self.predictor.trained and allocation_type == 'proactive':
            predicted_memory = self.predictor.predict_memory_usage(process)
            requested_memory = max(requested_memory, predicted_memory * 1.1)  # 10% buffer

        # Calculate required frames
        required_frames = max(1, int(np.ceil(requested_memory / self.page_size)))

        if len(self.free_frames) < required_frames:
            # Try to free memory using intelligent strategy
            freed = self._intelligent_memory_cleanup(required_frames)
            if not freed:
                return False, "Insufficient memory"

        # Allocate frames based on strategy
        if self.current_strategy == 'ai_optimized':
            allocated_frames = self._ai_optimized_allocation(pid, required_frames)
        else:
            allocated_frames = self._traditional_allocation(required_frames, self.current_strategy)

        if allocated_frames:
            self.allocated_frames[pid].update(allocated_frames)
            self.free_frames -= set(allocated_frames)
            self.performance_stats['successful_allocations'] += 1

            # Update process page table
            self._update_page_table(process, allocated_frames)

            self.allocation_history.append({
                'timestamp': datetime.now(),
                'pid': pid,
                'requested': requested_memory,
                'allocated_frames': len(allocated_frames),
                'strategy': self.current_strategy
            })

            return True, f"Allocated {len(allocated_frames)} frames"

        return False, "Allocation failed"

    def _ai_optimized_allocation(self, pid, required_frames):
        """AI-optimized frame allocation strategy"""
        process = self.processes[pid]

        # Consider process characteristics for allocation
        priority_weight = process.priority / 5.0
        cpu_intensity_weight = process.cpu_intensity

        # Prefer contiguous allocation for high-priority processes
        if priority_weight > 0.8:
            return self._allocate_contiguous_frames(required_frames)

        # For CPU-intensive processes, allocate frames closer together
        if cpu_intensity_weight > 0.7:
            return self._allocate_clustered_frames(required_frames)

        # Default to best-fit strategy
        return self._traditional_allocation(required_frames, 'best_fit')

    def _allocate_contiguous_frames(self, required_frames):
        """Allocate contiguous frames"""
        sorted_frames = sorted(self.free_frames)

        for i in range(len(sorted_frames) - required_frames + 1):
            # Check if we have required_frames contiguous frames
            contiguous = True
            for j in range(1, required_frames):
                if sorted_frames[i + j] != sorted_frames[i] + j:
                    contiguous = False
                    break

            if contiguous:
                return sorted_frames[i:i + required_frames]

        # Fallback to any available frames
        return list(sorted_frames)[:required_frames] if len(sorted_frames) >= required_frames else []

    def _allocate_clustered_frames(self, required_frames):
        """Allocate frames in clusters"""
        sorted_frames = sorted(self.free_frames)
        allocated = []

        # Try to allocate in chunks of 4 frames
        chunk_size = min(4, required_frames)
        remaining = required_frames

        i = 0
        while remaining > 0 and i < len(sorted_frames):
            chunk = min(chunk_size, remaining)
            if i + chunk <= len(sorted_frames):
                allocated.extend(sorted_frames[i:i + chunk])
                remaining -= chunk
                i += chunk + 1  # Leave small gap
            else:
                i += 1

        return allocated[:required_frames]

    def _traditional_allocation(self, required_frames, strategy):
        """Traditional allocation strategies"""
        available_frames = list(self.free_frames)

        if len(available_frames) < required_frames:
            return []

        if strategy == 'first_fit':
            return available_frames[:required_frames]

        elif strategy == 'best_fit':
            # For simplicity, just return required frames
            return available_frames[:required_frames]

        elif strategy == 'worst_fit':
            # Spread frames across memory
            step = max(1, len(available_frames) // required_frames)
            return available_frames[::step][:required_frames]

        return available_frames[:required_frames]

    def _update_page_table(self, process, allocated_frames):
        """Update process page table with allocated frames"""
        page_table = process.page_table
        frame_index = 0

        for page_num in range(len(allocated_frames)):
            if page_num < len(page_table.entries):
                entry = page_table.entries[page_num]
                entry.frame_number = allocated_frames[frame_index]
                entry.present = True
                frame_index += 1

    def _intelligent_memory_cleanup(self, required_frames):
        """Intelligent memory cleanup based on AI predictions"""
        # Find processes with predicted decrease in memory usage
        candidates = []

        for pid, process in self.processes.items():
            if self.predictor.trained:
                predicted_memory = self.predictor.predict_memory_usage(process)
                current_memory = process.memory_current

                if predicted_memory < current_memory * 0.8:  # 20% decrease predicted
                    priority_score = (current_memory - predicted_memory) / process.priority
                    candidates.append((pid, priority_score))

        # Sort by priority score (higher = better candidate for cleanup)
        candidates.sort(key=lambda x: x[1], reverse=True)

        freed_frames = 0
        for pid, _ in candidates:
            if freed_frames >= required_frames:
                break

            # Free some frames from this process
            process_frames = self.allocated_frames[pid]
            frames_to_free = min(len(process_frames) // 2, required_frames - freed_frames)

            if frames_to_free > 0:
                frames_list = list(process_frames)
                freed_frame_list = frames_list[:frames_to_free]

                self.allocated_frames[pid] -= set(freed_frame_list)
                self.free_frames.update(freed_frame_list)
                freed_frames += frames_to_free

        return freed_frames >= required_frames

    def deallocate_memory(self, pid):
        """Deallocate all memory for a process"""
        if pid in self.allocated_frames:
            freed_frames = self.allocated_frames[pid]
            self.free_frames.update(freed_frames)
            del self.allocated_frames[pid]

            if pid in self.processes:
                del self.processes[pid]

            return True, f"Deallocated {len(freed_frames)} frames"

        return False, "Process not found"

    def get_memory_statistics(self):
        """Get comprehensive memory statistics"""
        total_allocated = sum(len(frames) for frames in self.allocated_frames.values())
        memory_utilization = total_allocated / self.num_frames

        fragmentation = self._calculate_fragmentation()
        stats = {
            'total_memory_kb': self.total_memory,
            'total_frames': self.num_frames,
            'allocated_frames': total_allocated,
            'free_frames': len(self.free_frames),
            'memory_utilization': memory_utilization,
            'fragmentation_ratio': fragmentation,
            'active_processes': len(self.processes),
            'page_faults': sum(alg.page_faults for alg in self.page_algorithms.values() if alg),
            'prediction_accuracy': np.mean(self.performance_stats['prediction_accuracy']) if self.performance_stats['prediction_accuracy'] else 0
        }

        return stats

    def _calculate_fragmentation(self):
        """Calculate memory fragmentation ratio"""
        if not self.free_frames:
            return 0.0

        # Count free block fragments
        sorted_free = sorted(self.free_frames)
        fragments = 1

        for i in range(1, len(sorted_free)):
            if sorted_free[i] != sorted_free[i-1] + 1:
                fragments += 1

        # Fragmentation ratio: more fragments = higher fragmentation
        return fragments / len(sorted_free) if sorted_free else 0

PHASE 6: SIMULATION ENGINE & PERFORMANCE EVALUATOR

In [ ]:
class MemorySimulationEngine:
    """Advanced simulation engine để test và đánh giá hệ thống"""
    def __init__(self, memory_manager):
        self.memory_manager = memory_manager
        self.simulation_results = []
        self.benchmark_results = {}

    def create_realistic_workload(self, num_processes=50, simulation_time=500):
        """Tạo workload thực tế với nhiều loại process khác nhau"""
        processes = []

        # Các loại workload khác nhau
        workload_types = {
            'web_server': {'memory_base': 512, 'cpu_intensity': 0.6, 'pattern': 'periodic'},
            'database': {'memory_base': 2048, 'cpu_intensity': 0.8, 'pattern': 'linear'},
            'batch_job': {'memory_base': 1024, 'cpu_intensity': 0.9, 'pattern': 'burst'},
            'background_service': {'memory_base': 256, 'cpu_intensity': 0.3, 'pattern': 'random_walk'},
            'ml_training': {'memory_base': 4096, 'cpu_intensity': 0.95, 'pattern': 'burst'}
        }

        for i in range(num_processes):
            workload_type = random.choice(list(workload_types.keys()))
            config = workload_types[workload_type]

            process = Process(
                pid=i,
                memory_base=config['memory_base'] + random.randint(-100, 100),
                cpu_intensity=config['cpu_intensity'] + random.uniform(-0.1, 0.1)
            )

            # Override pattern type based on workload
            process.pattern_type = config['pattern']
            process.pattern_params = process._generate_pattern_params()

            processes.append(process)

        return processes

    def run_simulation(self, processes, simulation_time=500):
        """Chạy simulation chính với monitoring chi tiết"""
        print(f"Starting simulation with {len(processes)} processes for {simulation_time} steps...")

        # Register all processes
        for process in processes:
            self.memory_manager.register_process(process)

        # Collect training data first
        training_processes = processes[:len(processes)//2]  # Use half for training
        self.memory_manager.predictor.collect_training_data(training_processes, 100)
        self.memory_manager.predictor.train_models()

        simulation_log = []

        for step in range(simulation_time):
            step_stats = {
                'step': step,
                'timestamp': datetime.now() + timedelta(seconds=step),
                'processes': {}
            }

            # Simulate each process
            for process in processes:
                # Update memory usage
                actual_memory = process.simulate_memory_usage(step)

                # Get AI prediction
                if self.memory_manager.predictor.trained:
                    predicted_memory = self.memory_manager.predictor.predict_memory_usage(process)

                    # Calculate prediction accuracy
                    if actual_memory > 0:
                        accuracy = 1 - abs(predicted_memory - actual_memory) / actual_memory
                        self.memory_manager.performance_stats['prediction_accuracy'].append(accuracy)
                else:
                    predicted_memory = actual_memory

                # Memory allocation decisions
                allocation_needed = actual_memory > sum(4 for _ in self.memory_manager.allocated_frames.get(process.pid, []))

                if allocation_needed:
                    success, msg = self.memory_manager.allocate_memory(
                        process.pid,
                        actual_memory,
                        'proactive' if self.memory_manager.predictor.trained else 'dynamic'
                    )

                # Log process state
                step_stats['processes'][process.pid] = {
                    'actual_memory': actual_memory,
                    'predicted_memory': predicted_memory,
                    'allocated_frames': len(self.memory_manager.allocated_frames.get(process.pid, [])),
                    'lifecycle_stage': process.lifecycle_stage,
                    'pattern_type': process.pattern_type
                }

            # System-wide stats
            step_stats['system'] = self.memory_manager.get_memory_statistics()
            simulation_log.append(step_stats)

            # Progress indicator
            if step % 50 == 0:
                print(f"  Step {step}/{simulation_time} - Memory utilization: {step_stats['system']['memory_utilization']:.2%}")

        self.simulation_results = simulation_log
        print("Simulation completed!")
        return simulation_log

    def benchmark_algorithms(self, reference_string=None):
        """Benchmark các page replacement algorithms"""
        print("Benchmarking page replacement algorithms...")

        if reference_string is None:
            # Generate random reference string
            reference_string = [random.randint(0, 50) for _ in range(1000)]

        results = {}
        num_frames = 10

        # Test FIFO
        fifo = FIFOPageReplacement(num_frames)
        for page in reference_string:
            fifo.access_page(page)

        results['FIFO'] = {
            'page_faults': fifo.page_faults,
            'page_hits': fifo.page_hits,
            'hit_ratio': fifo.page_hits / (fifo.page_hits + fifo.page_faults)
        }

        # Test LRU
        lru = LRUPageReplacement(num_frames)
        for page in reference_string:
            lru.access_page(page)

        results['LRU'] = {
            'page_faults': lru.page_faults,
            'page_hits': lru.page_hits,
            'hit_ratio': lru.page_hits / (lru.page_hits + lru.page_faults)
        }

        # Test OPT
        opt = OPTPageReplacement(num_frames, reference_string)
        for page in reference_string:
            opt.access_page(page)

        results['OPT'] = {
            'page_faults': opt.page_faults,
            'page_hits': opt.page_hits,
            'hit_ratio': opt.page_hits / (opt.page_hits + opt.page_faults)
        }

        self.benchmark_results = results

        print("Algorithm Performance:")
        for alg_name, stats in results.items():
            print(f"  {alg_name}: {stats['page_faults']} faults, {stats['hit_ratio']:.2%} hit ratio")

        return results

PHASE 7: ADVANCED VISUALIZATION & ANALYSIS

In [ ]:
class MemoryAnalyzer:
    """Advanced analysis và visualization cho simulation results"""
    def __init__(self, simulation_results, benchmark_results=None):
        self.simulation_results = simulation_results
        self.benchmark_results = benchmark_results

    def create_comprehensive_analysis(self):
        """Tạo comprehensive analysis với multiple visualizations"""

        if not self.simulation_results:
            print("No simulation data available for analysis")
            return

        # Set up plotting style
        plt.style.use('seaborn-v0_8')
        fig = plt.figure(figsize=(20, 15))

        # 1. Memory Utilization Over Time
        ax1 = plt.subplot(3, 3, 1)
        steps = [step['step'] for step in self.simulation_results]
        utilization = [step['system']['memory_utilization'] for step in self.simulation_results]
        plt.plot(steps, utilization, linewidth=2, color='blue')
        plt.title('System Memory Utilization Over Time', fontsize=12, fontweight='bold')
        plt.xlabel('Simulation Step')
        plt.ylabel('Memory Utilization (%)')
        plt.grid(True, alpha=0.3)

        # 2. Memory Fragmentation
        ax2 = plt.subplot(3, 3, 2)
        fragmentation = [step['system']['fragmentation_ratio'] for step in self.simulation_results]
        plt.plot(steps, fragmentation, linewidth=2, color='red')
        plt.title('Memory Fragmentation Ratio', fontsize=12, fontweight='bold')
        plt.xlabel('Simulation Step')
        plt.ylabel('Fragmentation Ratio')
        plt.grid(True, alpha=0.3)

        # 3. Active Processes Count
        ax3 = plt.subplot(3, 3, 3)
        active_processes = [step['system']['active_processes'] for step in self.simulation_results]
        plt.plot(steps, active_processes, linewidth=2, color='green')
        plt.title('Active Processes Count', fontsize=12, fontweight='bold')
        plt.xlabel('Simulation Step')
        plt.ylabel('Number of Processes')
        plt.grid(True, alpha=0.3)

        # 4. Memory Usage Distribution by Process Type
        ax4 = plt.subplot(3, 3, 4)
        self._plot_process_type_distribution()

        # 5. Prediction Accuracy Analysis
        ax5 = plt.subplot(3, 3, 5)
        self._plot_prediction_accuracy()

        # 6. Page Replacement Algorithm Comparison
        ax6 = plt.subplot(3, 3, 6)
        if self.benchmark_results:
            self._plot_algorithm_comparison()

        # 7. Memory Allocation Patterns Heatmap
        ax7 = plt.subplot(3, 3, 7)
        self._plot_allocation_heatmap()

        # 8. Process Lifecycle Analysis
        ax8 = plt.subplot(3, 3, 8)
        self._plot_lifecycle_analysis()

        # 9. Performance Metrics Summary
        ax9 = plt.subplot(3, 3, 9)
        self._plot_performance_summary()

        plt.tight_layout()
        plt.savefig('memory_analysis_comprehensive.png', dpi=300, bbox_inches='tight')
        plt.show()

        # Generate detailed report
        self._generate_detailed_report()

    def _plot_process_type_distribution(self):
        """Plot memory usage distribution by process pattern type"""
        pattern_memory = defaultdict(list)

        for step in self.simulation_results[-50:]:  # Last 50 steps
            for pid, process_data in step['processes'].items():
                pattern_type = process_data['pattern_type']
                actual_memory = process_data['actual_memory']
                pattern_memory[pattern_type].append(actual_memory)

        # Create box plot
        data_for_plot = []
        labels = []
        for pattern, memories in pattern_memory.items():
            data_for_plot.append(memories)
            labels.append(pattern.replace('_', ' ').title())

        if data_for_plot:
            plt.boxplot(data_for_plot, labels=labels)
            plt.title('Memory Usage by Process Pattern', fontsize=12, fontweight='bold')
            plt.ylabel('Memory Usage (KB)')
            plt.xticks(rotation=45)

    def _plot_prediction_accuracy(self):
        """Plot AI prediction accuracy over time"""
        accuracies = []
        steps_with_predictions = []

        for step in self.simulation_results:
            step_accuracies = []
            for pid, process_data in step['processes'].items():
                actual = process_data['actual_memory']
                predicted = process_data['predicted_memory']
                if actual > 0:
                    accuracy = 1 - abs(predicted - actual) / actual
                    step_accuracies.append(max(0, min(1, accuracy)))  # Clamp between 0 and 1

            if step_accuracies:
                accuracies.append(np.mean(step_accuracies))
                steps_with_predictions.append(step['step'])

        if accuracies:
            plt.plot(steps_with_predictions, accuracies, linewidth=2, color='purple')
            plt.axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='80% Target')
            plt.title('AI Prediction Accuracy Over Time', fontsize=12, fontweight='bold')
            plt.xlabel('Simulation Step')
            plt.ylabel('Prediction Accuracy')
            plt.legend()
            plt.grid(True, alpha=0.3)

    def _plot_algorithm_comparison(self):
        """Plot page replacement algorithm comparison"""
        algorithms = list(self.benchmark_results.keys())
        hit_ratios = [self.benchmark_results[alg]['hit_ratio'] for alg in algorithms]

        bars = plt.bar(algorithms, hit_ratios, color=['skyblue', 'lightgreen', 'gold'])
        plt.title('Page Replacement Algorithm Hit Ratios', fontsize=12, fontweight='bold')
        plt.ylabel('Hit Ratio')
        plt.ylim(0, 1)

        # Add value labels on bars
        for bar, ratio in zip(bars, hit_ratios):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{ratio:.2%}', ha='center', va='bottom')

    def _plot_allocation_heatmap(self):
        """Plot memory allocation patterns as heatmap"""
        # Create allocation matrix
        time_steps = min(50, len(self.simulation_results))
        max_processes = max(len(step['processes']) for step in self.simulation_results[-time_steps:])

        allocation_matrix = np.zeros((time_steps, max_processes))

        for i, step in enumerate(self.simulation_results[-time_steps:]):
            for j, (pid, process_data) in enumerate(step['processes'].items()):
                if j < max_processes:
                    allocation_matrix[i, j] = process_data['allocated_frames']

        if allocation_matrix.size > 0:
            sns.heatmap(allocation_matrix, cmap='YlOrRd', cbar_kws={'label': 'Allocated Frames'})
            plt.title('Memory Allocation Heatmap', fontsize=12, fontweight='bold')
            plt.xlabel('Process ID')
            plt.ylabel('Time Step (Recent)')

    def _plot_lifecycle_analysis(self):
        """Analyze process lifecycle stages"""
        lifecycle_counts = defaultdict(int)

        for step in self.simulation_results[-10:]:  # Last 10 steps
            for pid, process_data in step['processes'].items():
                lifecycle_stage = process_data['lifecycle_stage']
                lifecycle_counts[lifecycle_stage] += 1

        if lifecycle_counts:
            stages = list(lifecycle_counts.keys())
            counts = list(lifecycle_counts.values())

            plt.pie(counts, labels=stages, autopct='%1.1f%%', startangle=90)
            plt.title('Process Lifecycle Distribution', fontsize=12, fontweight='bold')

    def _plot_performance_summary(self):
        """Create performance metrics summary"""
        final_stats = self.simulation_results[-1]['system'] if self.simulation_results else {}

        metrics = {
            'Memory\nUtilization': final_stats.get('memory_utilization', 0) * 100,
            'Fragmentation\nRatio': final_stats.get('fragmentation_ratio', 0) * 100,
            'Prediction\nAccuracy': final_stats.get('prediction_accuracy', 0) * 100
        }

        bars = plt.bar(metrics.keys(), metrics.values(),
                      color=['lightblue', 'lightcoral', 'lightgreen'])
        plt.title('Final Performance Metrics', fontsize=12, fontweight='bold')
        plt.ylabel('Percentage (%)')
        plt.ylim(0, 100)

        # Add value labels
        for bar, value in zip(bars, metrics.values()):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{value:.1f}%', ha='center', va='bottom')

    def _generate_detailed_report(self):
        """Generate detailed text report"""
        print("\n" + "="*80)
        print("INTELLIGENT MEMORY MANAGEMENT SYSTEM - DETAILED ANALYSIS REPORT")
        print("="*80)

        if not self.simulation_results:
            print("No simulation data available")
            return

        final_stats = self.simulation_results[-1]['system']

        print(f"\n SYSTEM OVERVIEW:")
        print(f"   Total Memory: {final_stats['total_memory_kb']:,} KB")
        print(f"   Total Frames: {final_stats['total_frames']:,}")
        print(f"   Memory Utilization: {final_stats['memory_utilization']:.2%}")
        print(f"   Active Processes: {final_stats['active_processes']}")
        print(f"   Fragmentation Ratio: {final_stats['fragmentation_ratio']:.3f}")

        # Process pattern analysis
        pattern_stats = defaultdict(list)
        for step in self.simulation_results[-20:]:
            for pid, process_data in step['processes'].items():
                pattern_stats[process_data['pattern_type']].append(process_data['actual_memory'])

        print(f"\n PROCESS PATTERN ANALYSIS:")
        for pattern, memories in pattern_stats.items():
            avg_memory = np.mean(memories)
            std_memory = np.std(memories)
            print(f"   {pattern.replace('_', ' ').title()}: Avg={avg_memory:.1f}KB, Std={std_memory:.1f}KB")

        # AI Performance
        if final_stats.get('prediction_accuracy', 0) > 0:
            print(f"\n AI PREDICTOR PERFORMANCE:")
            print(f"   Average Prediction Accuracy: {final_stats['prediction_accuracy']:.2%}")
            print(f"   Model Status: {'Trained and Active' if final_stats['prediction_accuracy'] > 0 else 'Not Trained'}")

        # Algorithm comparison
        if self.benchmark_results:
            print(f"\nPAGE REPLACEMENT ALGORITHM COMPARISON:")
            for alg_name, stats in self.benchmark_results.items():
                print(f"   {alg_name}: {stats['page_faults']} faults, {stats['hit_ratio']:.2%} hit ratio")

        print(f"\nAnalysis completed successfully!")
        print("="*80)

PHASE 8: MAIN EXCUTION & DEMO

In [ ]:
def main_demo():
    """Main demo function - Chạy toàn bộ hệ thống"""
    print("INTELLIGENT MEMORY MANAGEMENT SYSTEM")
    print("="*60)
    print("Team Lead: AI-Powered Memory Manager")
    print("Mission: Optimize memory allocation using Machine Learning")
    print("="*60)

    # Initialize system
    print("\nInitializing system components...")
    memory_manager = IntelligentMemoryManager(
        total_memory=8192,  # 8GB
        page_size=4,        # 4KB pages
        num_frames=2048     # 2048 frames
    )

    simulation_engine = MemorySimulationEngine(memory_manager)

    # Create realistic workload
    print("Creating realistic workload...")
    processes = simulation_engine.create_realistic_workload(
        num_processes=30,
        simulation_time=300
    )

    print(f"   Created {len(processes)} processes with various patterns:")
    pattern_count = {}
    for p in processes:
        pattern_count[p.pattern_type] = pattern_count.get(p.pattern_type, 0) + 1

    for pattern, count in pattern_count.items():
        print(f"     {pattern.replace('_', ' ').title()}: {count} processes")

    # Run simulation
    print(f"\nRunning simulation...")
    simulation_results = simulation_engine.run_simulation(processes, simulation_time=300)

    # Benchmark algorithms
    print(f"\nBenchmarking page replacement algorithms...")
    benchmark_results = simulation_engine.benchmark_algorithms()

    # Advanced analysis
    #print(f"\nPerforming advanced analysis...")
    #analyzer = MemoryAnalyzer(simulation_results, benchmark_results)
    #analyzer.create_comprehensive_analysis()

    # Final summary
    final_stats = memory_manager.get_memory_statistics()
    print(f"\n MISSION ACCOMPLISHED!")
    print(f"   Final Memory Utilization: {final_stats['memory_utilization']:.2%}")
    print(f"   Fragmentation Ratio: {final_stats['fragmentation_ratio']:.3f}")
    print(f"   Total Allocations: {memory_manager.performance_stats['total_allocations']}")
    print(f"   Success Rate: {memory_manager.performance_stats['successful_allocations']/max(1,memory_manager.performance_stats['total_allocations']):.2%}")

    return memory_manager, simulation_results, benchmark_results

if __name__ == "__main__":
    # Chạy demo chính
    manager, results, benchmarks = main_demo()

INTELLIGENT MEMORY MANAGEMENT SYSTEM
Team Lead: AI-Powered Memory Manager
Mission: Optimize memory allocation using Machine Learning

Initializing system components...
Creating realistic workload...
   Created 30 processes with various patterns:
     Random Walk: 4 processes
     Burst: 18 processes
     Periodic: 1 processes
     Linear: 7 processes

Running simulation...
Starting simulation with 30 processes for 300 steps...
Registered process 0 with pattern: random_walk
Registered process 1 with pattern: burst
Registered process 2 with pattern: random_walk
Registered process 3 with pattern: burst
Registered process 4 with pattern: random_walk
Registered process 5 with pattern: periodic
Registered process 6 with pattern: linear
Registered process 7 with pattern: random_walk
Registered process 8 with pattern: burst
Registered process 9 with pattern: burst
Registered process 10 with pattern: burst
Registered process 11 with pattern: burst
Registered process 12 with pattern: burst
Regis